#### Binary Classification with a Bank Churn Dataset  
Playground Series - Season 4, Episode 1

https://www.kaggle.com/competitions/playground-series-s4e1/data

In [2]:

import pandas as pd
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")


In [3]:
df = pd.read_csv("../data/playground-series-s4e1/train.csv")

# Display the first few rows of the dataset
df.head()

,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,0,15674932,Okwudilichukwu,668,France,Male,33.0,3,0.00,2,1.0,0.0,181449.97,0
1,1,15749177,Okwudiliolisa,627,France,Male,33.0,1,0.00,2,1.0,1.0,49503.50,0
2,2,15694510,Hsueh,678,France,Male,40.0,10,0.00,2,1.0,0.0,184866.69,0
3,3,15741417,Kao,581,France,Male,34.0,2,148882.54,1,1.0,1.0,84560.88,0
4,4,15766172,Chiemenam,716,Spain,Male,33.0,5,0.00,2,1.0,1.0,15068.83,0


In [4]:
df.shape

(165034, 14)

In [5]:
df.Exited.value_counts()

Exited
0    130113
1     34921
Name: count, dtype: int64

In [6]:
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Prepare the data
X = df.drop(columns=['Exited', 'id', 'CustomerId', 'Surname'])
y = df['Exited']

# Convert text columns to 'category' dtype
for col in ['Geography', 'Gender']:
    X[col] = X[col].astype('category')

# Set 'Geography' and 'Gender' as categorical features
categorical_features = ['Geography', 'Gender']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize the LightGBM classifier with categorical features
model = LGBMClassifier(
    n_estimators=4,
    learning_rate=0.1,
    categorical_feature=categorical_features,
    random_state=42,
    verbose=-1
)

# Train the model
model.fit(X_train, y_train, categorical_feature=categorical_features)
print
# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"4 trees Model Accuracy: {accuracy:.4f}")

# Get predicted probabilities for the positive class
y_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)
print(f"4 trees Model AUC: {auc:.4f}")


4 trees Model Accuracy: 0.7884
4 trees Model AUC: 0.8808


In [7]:

# Get the predicted probabilities for the positive class from the first model
init_score = model.predict(X_train, raw_score=True)

# Train a second LGBM model using the init_score
second_model = LGBMClassifier(
    n_estimators=2,
    learning_rate=0.1,
    categorical_feature=categorical_features,
    random_state=42,
    verbose=-1
)

# Pass init_score to fit method
second_model.fit(
    X_train,
    y_train,
    init_score=init_score
)

# Evaluate the second model
y_pred_second = second_model.predict(X_test)
accuracy_second = accuracy_score(y_test, y_pred_second)
print(f"Second +2 Trees Model Accuracy: {accuracy_second:.4f}")

# AUC for the second model
y_proba_second = second_model.predict_proba(X_test)[:, 1]
auc_second = roc_auc_score(y_test, y_proba_second)
print(f"Second +2 Trees Model AUC: {auc_second:.4f}")

Second +2 Trees Model Accuracy: 0.8142
Second +2 Trees Model AUC: 0.8793


In [8]:

# Initialize the LightGBM modelegressor
model_6trees = LGBMClassifier(
    n_estimators=6,
    learning_rate=0.1,
    categorical_feature=categorical_features,
    random_state=42,
    verbose=-1
)

# Train the model
model_6trees.fit(X_train, y_train)

# Predict on the test set
y_pred = model_6trees.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"6 Trees Model Accuracy: {accuracy:.4f}")

# Get predicted probabilities for the positive class
y_proba = model_6trees.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)
print(f"6 Trees Model AUC: {auc:.4f}")

6 Trees Model Accuracy: 0.8346
6 Trees Model AUC: 0.8819


In [9]:
from scipy.special import expit  # Sigmoid function

# 4. For prediction: sum raw scores from both models, then apply sigmoid
raw1 = model.predict(X_test, raw_score=True)
raw2 = second_model.predict(X_test, raw_score=True)
final_raw = raw1 + raw2
final_proba = expit(final_raw)  # This is the final predict_proba for the positive
final_pred = (final_proba >= 0.5).astype(int)

# Evaluate the combined model
combined_accuracy = accuracy_score(y_test, final_pred)
print(f"Combined Model Accuracy: {combined_accuracy:.4f}")

# AUC for the combined model
combined_auc = roc_auc_score(y_test, final_proba)
print(f"Combined Model AUC: {combined_auc:.4f}")

Combined Model Accuracy: 0.8346
Combined Model AUC: 0.8819
